# pyR0compute: ejemplos

Cálculo simbólico del número reproductivo básico $R_0$ mediante el método de la matriz de próxima generación. Basta con escribir el modelo de EDO e indicar qué compartimentos están infectados; todos los demás símbolos se reconocen automáticamente como parámetros.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yvan-BM-Git/pyR0compute/blob/main/examples/pyR0compute_ejemplos.ipynb)

In [1]:
%pip install -q pyR0compute    # Instala la librería desde PyPI (necesario en Colab o en un entorno nuevo)

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sympy as sp                            # Importa SymPy para realizar cálculo simbólico
from pyr0compute import R0Model               # Importa la clase R0Model desde el paquete pyr0compute
from IPython.display import Markdown, display # Importa Markdown y display para mostrar resultados en formato Markdown en Jupyter Notebook

## 1. Modelo SEIR con dinámica vital

Solo se entregan las ecuaciones y los compartimentos infectados.

In [3]:
seir = R0Model("""
    dS/dt = Lambda - beta*S*I - mu*S
    dE/dt = beta*S*I - (sigma + mu)*E
    dI/dt = sigma*E - (gamma + mu)*I
    dR/dt = gamma*I - mu*R
""", infected=["E", "I"])

seir.R0

Lambda*beta*sigma/(mu*(gamma + mu)*(mu + sigma))

In [4]:
print("Parámetros detectados:", seir.parameters)
print("Equilibrio libre de enfermedad:", seir.dfe)

Parámetros detectados: (Lambda, beta, gamma, mu, sigma)
Equilibrio libre de enfermedad: {S: Lambda/mu, E: 0, I: 0, R: 0}


Las matrices $F$, $V$ y $K = FV^{-1}$ están disponibles como objetos de SymPy:

In [5]:
seir.F, seir.V, seir.K

(Matrix([
 [0, Lambda*beta/mu],
 [0,              0]]),
 Matrix([
 [mu + sigma,          0],
 [    -sigma, gamma + mu]]),
 Matrix([
 [Lambda*beta*sigma/(mu*(gamma + mu)*(mu + sigma)), Lambda*beta/(mu*(gamma + mu))],
 [                                               0,                             0]]))

In [6]:
display(Markdown(seir.report_latex("markdown")))

## pyR0compute: next-generation matrix method

**Variables:** $S, E, I, R$  
**Infected compartments:** $E, I$  
**Parameters (detected automatically):** $\Lambda, \beta, \gamma, \mu, \sigma$

### Model

$$
\begin{aligned}
\frac{dS}{dt} &= - I S \beta + \Lambda - S \mu \\
\frac{dE}{dt} &= - E \left(\mu + \sigma\right) + I S \beta \\
\frac{dI}{dt} &= E \sigma - I \left(\gamma + \mu\right) \\
\frac{dR}{dt} &= I \gamma - R \mu
\end{aligned}
$$

### New-infection terms (automatic)

$$
\begin{aligned}
\mathcal{F}_{E} &= I S \beta \\
\mathcal{F}_{I} &= 0
\end{aligned}
$$

### Term classification

| Equation | Term | Class | Reason |
|---|---|:---:|---|
| $dE/dt$ | $E \left(- \mu - \sigma\right)$ | $\mathcal{V}$ | negative term |
| $dE/dt$ | $I S \beta$ | $\mathcal{F}$ | transfer from uninfected S |
| $dI/dt$ | $E \sigma$ | $\mathcal{V}$ | transfer from infected E |
| $dI/dt$ | $I \left(- \gamma - \mu\right)$ | $\mathcal{V}$ | negative term |

### Transition terms

$$
\begin{aligned}
\mathcal{V}_{E} &= E \mu + E \sigma \\
\mathcal{V}_{I} &= - E \sigma + I \gamma + I \mu
\end{aligned}
$$

### Disease-free equilibrium

$$
\begin{aligned}
S &= \frac{\Lambda}{\mu} \\
E &= 0 \\
I &= 0 \\
R &= 0
\end{aligned}
$$

### Next-generation matrix

$$
\begin{aligned}
F &= \begin{bmatrix}0 & \frac{\Lambda \beta}{\mu}\\0 & 0\end{bmatrix} \\
V &= \begin{bmatrix}\mu + \sigma & 0\\- \sigma & \gamma + \mu\end{bmatrix} \\
K = F V^{-1} &= \begin{bmatrix}\frac{\Lambda \beta \sigma}{\mu \left(\gamma + \mu\right) \left(\mu + \sigma\right)} & \frac{\Lambda \beta}{\mu \left(\gamma + \mu\right)}\\0 & 0\end{bmatrix}
\end{aligned}
$$

Only $E$ receive new infections, so $\mathcal{R}_0$ is the spectral radius of the next-generation matrix with small domain:

$$
\begin{aligned}
K_{\text{small}} &= \begin{bmatrix}\frac{\Lambda \beta \sigma}{\mu \left(\gamma + \mu\right) \left(\mu + \sigma\right)}\end{bmatrix}
\end{aligned}
$$

### Basic reproduction number

$$
\begin{aligned}
\mathcal{R}_0 = \rho\left(F V^{-1}\right) &= \frac{\Lambda \beta \sigma}{\mu \left(\gamma + \mu\right) \left(\mu + \sigma\right)}
\end{aligned}
$$

In [7]:
print(seir.report_latex())

\section*{pyR0compute: next-generation matrix method}

\textbf{Variables:} $S, E, I, R$ \\
\textbf{Infected compartments:} $E, I$ \\
\textbf{Parameters (detected automatically):} $\Lambda, \beta, \gamma, \mu, \sigma$

\subsection*{Model}

\begin{align*}
\frac{dS}{dt} &= - I S \beta + \Lambda - S \mu \\
\frac{dE}{dt} &= - E \left(\mu + \sigma\right) + I S \beta \\
\frac{dI}{dt} &= E \sigma - I \left(\gamma + \mu\right) \\
\frac{dR}{dt} &= I \gamma - R \mu
\end{align*}

\subsection*{New-infection terms (automatic)}

\begin{align*}
\mathcal{F}_{E} &= I S \beta \\
\mathcal{F}_{I} &= 0
\end{align*}

\subsection*{Term classification}

\begin{tabular}{llcl}
\hline
Equation & Term & Class & Reason \\
\hline
$dE/dt$ & $E \left(- \mu - \sigma\right)$ & $\mathcal{V}$ & negative term \\
$dE/dt$ & $I S \beta$ & $\mathcal{F}$ & transfer from uninfected S \\
$dI/dt$ & $E \sigma$ & $\mathcal{V}$ & transfer from infected E \\
$dI/dt$ & $I \left(- \gamma - \mu\right)$ & $\mathcal{V}$ & negative term \\


In [8]:
print(seir.latex())

\frac{\Lambda \beta \sigma}{\mu \left(\gamma + \mu\right) \left(\mu + \sigma\right)}


### Índices de sensibilidad

$\Upsilon_p = \dfrac{\partial R_0}{\partial p}\dfrac{p}{R_0}$: un aumento de 1% en $p$ modifica $R_0$ en $\Upsilon_p$%.

In [9]:
seir.sensitivity_indices()

{Lambda: 1,
 beta: 1,
 gamma: -gamma/(gamma + mu),
 mu: -(2*gamma*mu + gamma*sigma + 3*mu**2 + 2*mu*sigma)/((gamma + mu)*(mu + sigma)),
 sigma: mu/(mu + sigma)}

In [10]:
values = {"Lambda": 10, "beta": 0.001, "sigma": 0.2, "gamma": 0.1, "mu": 0.02}
print("R0 =", seir.R0_numeric(values))
seir.sensitivity_indices(values)

R0 = 3.787878787878788


{Lambda: 1.0,
 beta: 1.0,
 gamma: -0.8333333333333333,
 mu: -1.2575757575757578,
 sigma: 0.09090909090909093}

## 2. Modelo SIR clásico (población cerrada)

Sin nacimientos, el equilibrio libre de enfermedad no es único, por lo que se indica la población susceptible en ese equilibrio.

In [11]:
sir = R0Model({"S": "-beta*S*I/N",
               "I": "beta*S*I/N - gamma*I",
               "R": "gamma*I"}, infected=["I"], dfe={"S": "N"})
sir.R0

beta/gamma

Sin `dfe`, el mensaje de error indica exactamente qué falta:

In [12]:
try:
    R0Model({"S": "-beta*S*I/N", "I": "beta*S*I/N - gamma*I", "R": "gamma*I"}, infected=["I"])
except Exception as e:
    print(type(e).__name__, "->", e)

DiseaseFreeEquilibriumError -> The disease-free value of ['S'] is not determined by the equations (e.g. a closed population without births). Provide it with dfe={'S': ...}.


## 3. Enfermedad transmitida por vectores: modelo de Ross-Macdonald

In [13]:
rm = R0Model("""
    dS_h/dt = L_h - a*b_h*S_h*I_v/N_h - mu_h*S_h
    dI_h/dt = a*b_h*S_h*I_v/N_h - (gamma + mu_h)*I_h
    dR_h/dt = gamma*I_h - mu_h*R_h
    dS_v/dt = L_v - a*b_v*S_v*I_h/N_h - mu_v*S_v
    dI_v/dt = a*b_v*S_v*I_h/N_h - mu_v*I_v
""", infected=["I_h", "I_v"])
rm.new_infections

{I_h: I_v*S_h*a*b_h/N_h, I_v: I_h*S_v*a*b_v/N_h}

In [14]:
rm.K

Matrix([
[                                  0, L_h*a*b_h/(N_h*mu_h*mu_v)],
[L_v*a*b_v/(N_h*mu_v*(gamma + mu_h)),                         0]])

In [15]:
rm.R0

a*sqrt(L_h*L_v*b_h*b_v)/(N_h*mu_v*sqrt(mu_h*(gamma + mu_h)))

## 4. Modelo huésped-vector con transmisión humano-humano y vectores con crecimiento logístico

La población humana total se define como expresión auxiliar. El modelo tiene dos equilibrios libres de enfermedad (con y sin vectores); se elige el que tiene vectores y ambos quedan disponibles en `hv.dfe_candidates`.

In [16]:
hv = R0Model("""
    N_h = S_h + E_h + I_h + R_h
    dS_h/dt = Delta_h - delta*alpha_hv*S_h*I_v/N_h - beta*alpha_hh*S_h*I_h/N_h - (theta + mu_h)*S_h
    dE_h/dt = delta*alpha_hv*S_h*I_v/N_h + beta*alpha_hh*S_h*I_h/N_h - (sigma + mu_h)*E_h
    dI_h/dt = sigma*E_h - (gamma + mu_h)*I_h
    dR_h/dt = gamma*I_h - mu_h*R_h + theta*S_h
    dM_v/dt = phi_v*M_v*(1 - M_v/kappa_v) - (omega + mu_m)*M_v
    dS_v/dt = omega*M_v - delta*alpha_vh*S_v*I_h/N_h - (mu_v + zeta)*S_v
    dE_v/dt = delta*alpha_vh*S_v*I_h/N_h - (psi + mu_v + zeta)*E_v
    dI_v/dt = psi*E_v - (mu_v + zeta)*I_v
""", infected=["E_h", "I_h", "E_v", "I_v"])
hv.dfe

/tmp/ipykernel_189/135047437.py:1: UserWarning: 2 disease-free equilibria were found; using the one with the most non-zero compartments: {'M_v': -kappa_v*(mu_m + omega - phi_v)/phi_v, 'R_h': Delta_h*theta/(mu_h**2 + mu_h*theta), 'S_h': Delta_h/(mu_h + theta), 'S_v': -kappa_v*omega*(mu_m + omega - phi_v)/(phi_v*(mu_v + zeta))}. Pass dfe={...} to choose another one (all are in model.dfe_candidates).
  hv = R0Model("""


{S_h: Delta_h/(mu_h + theta),
 E_h: 0,
 I_h: 0,
 R_h: Delta_h*theta/(mu_h*(mu_h + theta)),
 M_v: kappa_v*(-mu_m - omega + phi_v)/phi_v,
 S_v: kappa_v*omega*(-mu_m - omega + phi_v)/(phi_v*(mu_v + zeta)),
 E_v: 0,
 I_v: 0}

Solo $E_h$ y $E_v$ reciben nuevas infecciones, por lo que $R_0$ es el radio espectral de la matriz de próxima generación de dominio pequeño, de tamaño $2\times 2$:

In [17]:
hv.next_generation_matrix_small

Matrix([
[                                                    alpha_hh*beta*mu_h*sigma/((gamma + mu_h)*(mu_h + sigma)*(mu_h + theta)), alpha_hv*delta*mu_h*psi/((mu_h + theta)*(mu_v + zeta)*(mu_v + psi + zeta))],
[-alpha_vh*delta*kappa_v*mu_h*omega*sigma*(mu_m + omega - phi_v)/(Delta_h*phi_v*(gamma + mu_h)*(mu_h + sigma)*(mu_v + zeta)),                                                                          0]])

In [18]:
hv.R0

alpha_hh*beta*mu_h*sigma/(2*(gamma + mu_h)*(mu_h + sigma)*(mu_h + theta)) + sqrt(alpha_hh**2*beta**2*mu_h**2*sigma**2/((gamma + mu_h)**2*(mu_h + sigma)**2*(mu_h + theta)**2) - 4*alpha_hv*alpha_vh*delta**2*kappa_v*mu_h**2*omega*psi*sigma*(mu_m + omega - phi_v)/(Delta_h*phi_v*(gamma + mu_h)*(mu_h + sigma)*(mu_h + theta)*(mu_v + zeta)**2*(mu_v + psi + zeta)))/2

## 5. Modelo intrahuésped: células blanco, células infectadas y virus

In [19]:
tiv = R0Model("""
    dT/dt = s - d*T - beta*T*V
    dI/dt = beta*T*V - delta*I
    dV/dt = p*I - c*V
""", infected=["I", "V"])
tiv.R0

beta*p*s/(c*d*delta)

## 6. van den Driessche y Watmough (2002), modelo con tratamiento (§4.1)

El término $p r_2 I$ (falla del tratamiento) se clasifica correctamente como una transición y no como una nueva infección.

In [20]:
tb = R0Model("""
    N = S + E + I + T
    dE/dt = beta1*S*I/N + beta2*T*I/N - (d + nu + r1)*E + p*r2*I
    dI/dt = nu*E - (d + r2)*I
    dS/dt = b - d*S - beta1*S*I/N
    dT/dt = -d*T + r1*E + q*r2*I - beta2*T*I/N
""", infected=["E", "I"])
tb.R0

-beta1*nu/(-d**2 - d*nu - d*r1 - d*r2 + nu*p*r2 - nu*r2 - r1*r2)

## 7. Dos cepas con superinfección

La cepa dominante depende de los parámetros, por lo que $R_0$ es el máximo de los números reproductivos de cada cepa.

In [21]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    two = R0Model("""
        dI1/dt = beta1*I1*S - (b + gamma1)*I1 + nu*I1*I2
        dI2/dt = beta2*I2*S - (b + gamma2)*I2 - nu*I1*I2
        dS/dt = b - b*S + gamma1*I1 + gamma2*I2 - (beta1*I1 + beta2*I2)*S
    """, infected=["I1", "I2"])
    R0 = two.R0
R0

Max(beta1/(b + gamma1), beta2/(b + gamma2))

## 8. Tu propio modelo

Reemplaza las ecuaciones y los compartimentos infectados. Si la separación automática entre nuevas infecciones y transiciones no es la que necesitas, usa `new_infections={...}`; para fijar el equilibrio libre de enfermedad, usa `dfe={...}`.

In [22]:
model = R0Model("""
    dS/dt = Lambda - beta*S*I - mu*S
    dI/dt = beta*S*I - (gamma + mu)*I
""", infected=["I"])
model.R0

Lambda*beta/(mu*(gamma + mu))

In [23]:
print(model.latex())

\frac{\Lambda \beta}{\mu \left(\gamma + \mu\right)}
